# TCN Filtered Baseline

TCN 기준점 모델. Pipeline D 필터링 데이터 기반 baseline을 학습합니다.

이 노트북은 **배정된 1개 기준점 실험만** 실행합니다. 별도 `SESSION_ID`나 외부 JSON 설정 파일을 사용하지 않습니다.

고정 조건:

- Experiment ID: `B-TCN-D`
- Model: `TCN`
- Preprocessing: `PP-D`
- Dataset: 이 노트북 변수 `TRAIN_CSV`, `VAL_CSV`, `TEST_CSV`에 직접 명시
- Window: `5s~9s`, `60 steps`
- Output: `results/baselines_phase0/B-TCN-D/`


## 1. Drive 마운트와 프로젝트 루트

공유 폴더 경로가 자동 탐색되지 않으면 `PROJECT_ROOT_OVERRIDE`에 직접 경로를 입력합니다.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive mount skipped.')

PROJECT_ROOT_OVERRIDE = ""
PROJECT_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Graduate-Project/Falling-Model-Development'),
    Path('/content/drive/MyDrive/Falling-Model-Development'),
    Path('/content/drive/MyDrive/졸업 과제/Falling-Model-Development'),
    Path('/content/drive/MyDrive/졸업 과제/Falling-Detection-Development'),
    Path.cwd(),
]

if PROJECT_ROOT_OVERRIDE.strip():
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser()
else:
    PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'scripts' / 'train_baseline.py').exists()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Project root not found. Set PROJECT_ROOT_OVERRIDE.')

os.chdir(PROJECT_ROOT)

# Experiment identity
EXPERIMENT_ID = 'B-TCN-D'
MODEL_TYPE = 'tcn'
PREPROCESSING = 'filtered'

# Dataset paths are intentionally declared in this notebook.
TRAIN_CSV = PROJECT_ROOT / 'dataset' / 'train.csv'
VAL_CSV = PROJECT_ROOT / 'dataset' / 'val.csv'
TEST_CSV = PROJECT_ROOT / 'dataset' / 'test.csv'

# Input/window policy
INPUT_MODE = 'split_csv'
FEATURE_SET = 'kp12'
LABEL_COLUMN = 'label'
POSITIVE_LABELS = [1]
LABEL_MODE = 'segment_max'
DATA_SCOPE = 'no_by'
WINDOW_START_SEC = 5.0
WINDOW_END_SEC = 9.0
TARGET_STEPS = 60
TRAIN_POSITIVE_STRIDE = 1
TRAIN_NEGATIVE_STRIDE = 5
EVAL_STRIDE = 1

# Training policy
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 0.001
SEED = 42
DROPOUT_RATE = 0.2
EARLY_STOP_PATIENCE = 10
THRESHOLD_COUNT = 19
MIN_VAL_RECALL = 0.0

# Model structure
TCN_CHANNELS = [32, 32, 64, 96]
TCN_DILATIONS = [1, 2, 4, 8]
TCN_KERNEL_SIZE = 3
GRU_UNITS = [64, 32]

# Quantization/evaluation policy
REPRESENTATIVE_SAMPLES = 256
QUANT_EVAL_MAX_WINDOWS = 5000

OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'baselines_phase0'
RESULT_DIR = OUTPUT_ROOT / EXPERIMENT_ID

print('PROJECT_ROOT  =', PROJECT_ROOT)
print('EXPERIMENT_ID =', EXPERIMENT_ID)
print('TRAIN_CSV     =', TRAIN_CSV)
print('VAL_CSV       =', VAL_CSV)
print('TEST_CSV      =', TEST_CSV)
print('RESULT_DIR    =', RESULT_DIR)


## 2. 런타임 준비

초기 Colab 런타임에서 필요한 패키지를 설치합니다.


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow', 'pandas', 'scikit-learn', 'matplotlib'],
    check=False,
)

import pandas as pd
from IPython.display import Image, Markdown, display
print('Runtime ready.')


## 3. 고정 실험 설정 확인

아래 표는 이 노트북이 실행할 단일 기준점 실험 설정입니다.


In [ ]:
EXPERIMENT_SUMMARY = {
    'experiment_id': EXPERIMENT_ID,
    'model_type': MODEL_TYPE,
    'preprocessing': PREPROCESSING,
    'input_mode': INPUT_MODE,
    'train_csv': str(TRAIN_CSV.relative_to(PROJECT_ROOT)),
    'val_csv': str(VAL_CSV.relative_to(PROJECT_ROOT)),
    'test_csv': str(TEST_CSV.relative_to(PROJECT_ROOT)),
    'feature_set': FEATURE_SET,
    'data_scope': DATA_SCOPE,
    'window_start_sec': WINDOW_START_SEC,
    'window_end_sec': WINDOW_END_SEC,
    'target_steps': TARGET_STEPS,
    'train_positive_stride': TRAIN_POSITIVE_STRIDE,
    'train_negative_stride': TRAIN_NEGATIVE_STRIDE,
    'eval_stride': EVAL_STRIDE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
}
display(pd.DataFrame([EXPERIMENT_SUMMARY]))


## 4. 학습 실행

Keras 학습 진행률과 TFLite/INT8 export 로그가 셀 출력에 그대로 남습니다.

테스트만 빠르게 확인하려면 아래 셀의 `SMOKE = False`를 `True`로 바꾸면 됩니다.


In [ ]:
SMOKE = False
EXPORT_TFLITE = True

def csv_ints(values):
    return ','.join(str(value) for value in values)

def run_streaming(cmd):
    print(' '.join(str(part) for part in cmd))
    process = subprocess.Popen([str(part) for part in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Command failed: {code}')

cmd = [
    sys.executable, 'scripts/train_baseline.py',
    '--project-root', PROJECT_ROOT,
    '--output-root', OUTPUT_ROOT,
    '--experiment-id', EXPERIMENT_ID,
    '--model-type', MODEL_TYPE,
    '--preprocessing', PREPROCESSING,
    '--input-mode', INPUT_MODE,
    '--train-csv', TRAIN_CSV,
    '--val-csv', VAL_CSV,
    '--test-csv', TEST_CSV,
    '--feature-set', FEATURE_SET,
    '--label-column', LABEL_COLUMN,
    '--positive-labels', csv_ints(POSITIVE_LABELS),
    '--label-mode', LABEL_MODE,
    '--data-scope', DATA_SCOPE,
    '--window-start-sec', WINDOW_START_SEC,
    '--window-end-sec', WINDOW_END_SEC,
    '--target-steps', TARGET_STEPS,
    '--train-positive-stride', TRAIN_POSITIVE_STRIDE,
    '--train-negative-stride', TRAIN_NEGATIVE_STRIDE,
    '--eval-stride', EVAL_STRIDE,
    '--batch-size', BATCH_SIZE,
    '--epochs', EPOCHS,
    '--learning-rate', LEARNING_RATE,
    '--seed', SEED,
    '--dropout-rate', DROPOUT_RATE,
    '--early-stop-patience', EARLY_STOP_PATIENCE,
    '--threshold-count', THRESHOLD_COUNT,
    '--min-val-recall', MIN_VAL_RECALL,
    '--tcn-channels', csv_ints(TCN_CHANNELS),
    '--tcn-dilations', csv_ints(TCN_DILATIONS),
    '--tcn-kernel-size', TCN_KERNEL_SIZE,
    '--gru-units', csv_ints(GRU_UNITS),
    '--representative-samples', REPRESENTATIVE_SAMPLES,
    '--quant-eval-max-windows', QUANT_EVAL_MAX_WINDOWS,
]
if SMOKE:
    cmd.append('--smoke')
if not EXPORT_TFLITE:
    cmd.append('--no-export-tflite')
run_streaming(cmd)


## 5. 성능 테이블

Float와 INT8 평가가 성공하면 `test_float`, `test_int8` 지표가 함께 표시됩니다.


In [ ]:
metrics = json.loads((RESULT_DIR / 'metrics.json').read_text())
rows = []
for key, item in metrics['metrics'].items():
    rows.append({
        'split_runtime': key,
        'accuracy': item.get('accuracy'),
        'precision': item.get('precision'),
        'recall': item.get('recall'),
        'f1': item.get('f1'),
        'auc_roc': item.get('auc_roc'),
        'pr_auc': item.get('pr_auc'),
        'positive_support': item.get('positive_support'),
    })
display(pd.DataFrame(rows))
display(Markdown(f"Selected threshold: `{metrics['threshold_selection']['threshold']:.3f}`"))
display(pd.DataFrame(metrics['threshold_selection']['sweep']))


## 6. 시각화 산출물

학습 곡선, threshold sweep, window 분포, float 평가 plot, INT8 평가 plot을 표시합니다.


In [ ]:
for filename in [
    'training_curve.png',
    'threshold_sweep.png',
    'window_distribution.png',
    'confusion_matrix.png',
    'roc_curve.png',
    'pr_curve.png',
    'int8_confusion_matrix.png',
    'int8_roc_curve.png',
    'int8_pr_curve.png',
]:
    path = RESULT_DIR / filename
    if path.exists():
        display(Markdown(f'### {filename}'))
        display(Image(filename=str(path)))
    else:
        print('Missing:', path)


## 7. 산출물 위치

이 노트북의 모든 결과는 아래 폴더에 저장됩니다.


In [ ]:
print(RESULT_DIR)
for path in sorted(RESULT_DIR.glob('*')):
    print(path.name)
